# Long-Horizon Emissions Forecasting for 2030 Target Assessment: A Comparative Study of N-HiTS, XGBoost, and Bayesian Models in Fast-Moving Consumer Goods Supply Chains
**Authors**: Idris Alugo 

**Paper**: *Applied Energy* (submitted 2026)  
**Zenodo DOI**: [to be assigned]

This notebook orchestrates the full pipeline:
1. Data loading & feature engineering  
2. Per-facility model training (N-HiTS, XGBoost-Quantile, BNN)  
3. Out-of-fold meta-feature construction  
4. Stacking ensemble (meta-learner) training  
5. 2030 forecast generation  
6. Risk & compliance assessment  
7. SHAP explainability  
8. Publication figures

All heavy logic lives in `src/` — this notebook is the **single entry point** to reproduce all results.


## 0. Environment

In [ ]:
import sys, importlib
print(f"Python {sys.version}")

required = [
    "numpy", "pandas", "torch", "xgboost",
    "neuralforecast", "shap", "sklearn",
    "matplotlib", "seaborn", "scipy",
]
missing = []
for pkg in required:
    try:
        importlib.import_module(pkg)
        print(f"  ✓ {pkg}")
    except ImportError:
        missing.append(pkg)
        print(f"  ✗ {pkg}  ← MISSING")

if missing:
    raise ImportError(f"Install missing packages: pip install {' '.join(missing)}")
else:
    print("\nAll dependencies satisfied.")


## 1. Imports & Logging

In [ ]:
import logging
import warnings
import numpy as np
import pandas as pd

# src modules
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # adjust if running from notebooks/

from config import (
    DATA_PATH, TARGET_COL, TEST_MONTHS, RANDOM_SEED,
    BASELINE_YEAR, TARGET_REDUCTION_RATE, MIN_TRAIN_ROWS,
)
from src.preprocessing import (
    load_and_clean_data,
    create_global_features_with_hierarchy,
)
from src.models.nhits_model   import run_nhits, predict_nhits_2030, shap_nhits
from src.models.xgboost_model import run_xgboost_quantile, predict_xgboost_2030, shap_xgboost
from src.models.bnn_model     import run_bnn, mc_predict_bnn, shap_bnn
from src.models.meta_learner  import (
    build_meta_features, train_meta_learner,
    predict_ensemble, evaluate_meta_learner,
)
from src.evaluation import (
    prepare_facility_targets,
    compute_calibrated_probabilities,
    compute_stakeholder_metrics,
    compare_model_metrics,
    compute_regional_summary,
    aggregate_shap_records,
)
from src.visualization import plot_all

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("pipeline")
np.random.seed(RANDOM_SEED)
logger.info("Imports complete.")


## 2. Data Loading & Feature Engineering

In [ ]:
# Load and clean raw data
df_raw = load_and_clean_data(DATA_PATH)
logger.info("Raw data: %s", df_raw.shape)

# Define base numeric features (before hierarchy/lag enrichment)
BASE_NUMERIC_FEATURES = [
    "Month", "Year", "DayOfYear",
    "Production", "Energy_MWh", "Waste_Kg",
    "Renewable_percent", "PolicyWeight",
]
region_cols = [c for c in df_raw.columns if c.startswith("Region_")]
base_features = BASE_NUMERIC_FEATURES + region_cols

# Create enhanced feature set (lags, rolling stats, hierarchy embeddings)
df, global_scalers, features = create_global_features_with_hierarchy(df_raw, base_features)

logger.info("Enhanced data: %s | Features: %d", df.shape, len(features))
logger.info("Facilities: %s", df["Facility"].nunique())
df.head(3)

## 3. Per-Facility Model Training

> Trains N-HiTS, XGBoost-Quantile, and BNN for each facility. Collects OOF predictions for meta-learning.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

results          = []
meta_feature_records = []
shap_records     = []
bnn_res_dict     = {}
xgb_res_dict     = {}
nhits_res_dict   = {}

tscv = TimeSeriesSplit(n_splits=3)

for fac in df["Facility"].unique():
    df_fac = df[df["Facility"] == fac].sort_values("Date").reset_index(drop=True)
    logger.info("=" * 60)
    logger.info("Facility: %s  (%d rows)", fac, len(df_fac))

    if len(df_fac) < MIN_TRAIN_ROWS or df_fac[TARGET_COL].nunique() < 2:
        logger.warning("Skipping %s — insufficient data or no target variance.", fac)
        continue

    nhits_oof = np.full(len(df_fac), np.nan)
    xgb_oof   = np.full(len(df_fac), np.nan)
    bnn_oof   = np.full(len(df_fac), np.nan)

    # ── Out-of-fold predictions (TimeSeriesSplit) ──────────────────────────
    for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(df_fac)):
        df_train = df_fac.iloc[train_idx]
        df_test  = df_fac.iloc[test_idx]
        n_test   = len(df_test)

        # N-HiTS OOF
        try:
            res = run_nhits(
                pd.concat([df_train, df_test], ignore_index=True),
                fac, features, TARGET_COL, test_months=n_test,
            )
            if res and res.get("model") is not None:
                for i, idx in enumerate(test_idx):
                    if i < len(res["y_pred"]):
                        nhits_oof[idx] = res["y_pred"][i]
        except Exception as e:
            logger.debug("N-HiTS OOF fold %d failed for %s: %s", fold_idx, fac, e)

        # XGBoost OOF
        try:
            res = run_xgboost_quantile(
                pd.concat([df_train, df_test], ignore_index=True),
                fac, features, TARGET_COL, test_months=n_test,
            )
            if res and res.get("models") and 0.5 in res["models"]:
                preds = res["models"][0.5].predict(df_test[features].values)
                for i, idx in enumerate(test_idx):
                    if i < len(preds):
                        xgb_oof[idx] = preds[i]
        except Exception as e:
            logger.debug("XGBoost OOF fold %d failed for %s: %s", fold_idx, fac, e)

        # BNN OOF
        try:
            res = run_bnn(
                pd.concat([df_train, df_test], ignore_index=True),
                fac, features, TARGET_COL, test_months=n_test, n_epochs=100,
            )
            if res and res.get("model") is not None:
                mc = mc_predict_bnn(res["model"], df_test[features].values)
                preds = mc.mean(axis=0)
                for i, idx in enumerate(test_idx):
                    if i < len(preds):
                        bnn_oof[idx] = preds[i]
        except Exception as e:
            logger.debug("BNN OOF fold %d failed for %s: %s", fold_idx, fac, e)

    # ── Final models trained on full facility data ─────────────────────────
    nhits_res = run_nhits(df_fac, fac, features, TARGET_COL, TEST_MONTHS)
    xgb_res   = run_xgboost_quantile(df_fac, fac, features, TARGET_COL, TEST_MONTHS)
    bnn_res   = run_bnn(df_fac, fac, features, TARGET_COL, TEST_MONTHS, n_epochs=300)

    nhits_res_dict[fac] = nhits_res
    xgb_res_dict[fac]   = xgb_res
    bnn_res_dict[fac]   = bnn_res

    results.append({
        "Facility":    fac,
        "NHITSMAPE":   nhits_res.get("mape", np.nan),
        "NHITSRMSE":   nhits_res.get("rmse", np.nan),
        "XGBoostMAPE": xgb_res.get("mape", np.nan),
        "XGBoostRMSE": xgb_res.get("rmse", np.nan),
        "BNNMAPE":     bnn_res.get("mape", np.nan),
        "BNNRMSE":     bnn_res.get("rmse", np.nan),
    })

    # ── Collect OOF meta-feature rows ─────────────────────────────────────
    for i in range(len(df_fac)):
        has_nhits = not np.isnan(nhits_oof[i])
        has_xgb   = not np.isnan(xgb_oof[i])
        has_bnn   = not np.isnan(bnn_oof[i])
        if has_nhits or has_xgb or has_bnn:
            meta_feature_records.append({
                "Facility": fac,
                "NHITS_OOF": nhits_oof[i] if has_nhits else 0.0,
                "XGB_OOF":   xgb_oof[i]   if has_xgb   else 0.0,
                "BNN_OOF":   bnn_oof[i]   if has_bnn   else 0.0,
                "y_true":    df_fac.iloc[i][TARGET_COL],
            })

    # ── SHAP analysis ──────────────────────────────────────────────────────
    X_val = df_fac[features].values[-TEST_MONTHS:]
    X_train_shap = df_fac[features].values[:-TEST_MONTHS]

    mean_abs_xgb = mean_abs_bnn = mean_abs_nhits = None

    if xgb_res and xgb_res.get("models") and 0.5 in xgb_res["models"]:
        try:
            sv = shap_xgboost(xgb_res["models"][0.5], X_val, features, fac, show_plot=False)
            if sv is not None:
                mean_abs_xgb = np.abs(sv).mean(axis=0)
        except Exception as e:
            logger.debug("XGB SHAP failed for %s: %s", fac, e)

    if bnn_res and bnn_res.get("model") is not None:
        try:
            sv = shap_bnn(bnn_res["model"], X_train_shap, X_val, features, fac, show_plot=False)
            if sv is not None:
                mean_abs_bnn = np.abs(sv).mean(axis=0)
        except Exception as e:
            logger.debug("BNN SHAP failed for %s: %s", fac, e)

    try:
        sv = shap_nhits(X_train_shap, X_val, features, fac, show_plot=False)
        if sv is not None:
            mean_abs_nhits = np.abs(sv).mean(axis=0)
    except Exception as e:
        logger.debug("N-HiTS SHAP failed for %s: %s", fac, e)

    if any(x is not None for x in [mean_abs_xgb, mean_abs_bnn, mean_abs_nhits]):
        zeros = np.zeros(len(features))
        for i, feat in enumerate(features):
            shap_records.append({
                "Facility":        fac,
                "Feature":         feat,
                "MeanAbsSHAPNHITS": mean_abs_nhits[i] if mean_abs_nhits is not None else zeros[i],
                "MeanAbsSHAPXGB":   mean_abs_xgb[i] if mean_abs_xgb is not None else zeros[i],
                "MeanAbsSHAPBNN":   mean_abs_bnn[i] if mean_abs_bnn is not None else zeros[i],
            })

results_df = pd.DataFrame(results)
logger.info("Training complete. %d facilities.", len(results_df))
results_df[["Facility","NHITSMAPE","XGBoostMAPE","BNNMAPE"]]

## 4. Meta-Learner Ensemble Training

In [ ]:
import pandas as pd

if "meta_feature_records" not in globals():
    raise RuntimeError("Run Cell 8 (Per-Facility Model Training) before this cell.")

meta_df = pd.DataFrame(meta_feature_records)
logger.info("Meta-feature rows collected: %d", len(meta_df))

# Augment with uncertainty & agreement features
meta_df_enhanced = build_meta_features(meta_df, bnn_res_dict, xgb_res_dict)

# Train stacking ensemble
meta_models, meta_feature_cols, meta_weights = train_meta_learner(
    meta_df_enhanced, verbose=True
)
logger.info("Meta-learner weights: %s", meta_weights)

# OOF evaluation
oof_eval = evaluate_meta_learner(
    meta_models, meta_weights, meta_feature_cols,
    meta_df_enhanced,
    individual_oof_cols={"N-HiTS": "NHITS_OOF", "XGBoost": "XGB_OOF", "BNN": "BNN_OOF"},
)
logger.info("Ensemble OOF MAPE=%.2f%%  RMSE=%.4f", oof_eval["ensemble_mape"], oof_eval["ensemble_rmse"])
pd.DataFrame([oof_eval]).T


## 5. 2030 Forecast Generation

In [ ]:
from src.preprocessing import prepare_2030_features

nhits_quantile_preds = {}
xgb_quantile_preds   = {}
bnn_mc_preds         = {}

for fac in results_df["Facility"].unique():
    df_fac = df[df["Facility"] == fac].copy()
    X_2030 = prepare_2030_features(df, fac, features, global_scalers, base_features)

    # N-HiTS
    nhits_res = nhits_res_dict.get(fac)
    if nhits_res and nhits_res.get("model") is not None:
        preds = predict_nhits_2030(nhits_res["model"], df_fac, fac, features)
        if preds:
            nhits_quantile_preds[fac] = preds

    # XGBoost
    xgb_res = xgb_res_dict.get(fac)
    if xgb_res and xgb_res.get("models"):
        preds = predict_xgboost_2030(xgb_res["models"], X_2030)
        if preds:
            xgb_quantile_preds[fac] = preds

    # BNN
    bnn_res = bnn_res_dict.get(fac)
    if bnn_res and bnn_res.get("model") is not None:
        samples = mc_predict_bnn(bnn_res["model"], X_2030)
        samples = np.clip(samples, 0, None)
        bnn_mc_preds[fac] = samples.flatten()

logger.info(
    "2030 forecasts: N-HiTS=%d  XGBoost=%d  BNN=%d",
    len(nhits_quantile_preds), len(xgb_quantile_preds), len(bnn_mc_preds),
)


## 6. Risk & Compliance Assessment

In [ ]:
# Facility-level baselines and targets
facility_baseline = prepare_facility_targets(
    df_raw,
    baseline_year=BASELINE_YEAR,
    target_reduction=TARGET_REDUCTION_RATE,
)

# Per-facility compliance probabilities
risk_df = compute_calibrated_probabilities(
    facility_baseline,
    nhits_quantile_preds,
    xgb_quantile_preds,
    bnn_mc_preds,
    results_df,
    n_bootstrap=1000,
)

# Regional roll-up
regional_summary = compute_regional_summary(risk_df, facility_baseline)

# Model comparison
comparison_df = compare_model_metrics(results_df)

print("\n── Risk Distribution ──────────────────────────")
print(risk_df["RiskLevelEnsemble"].value_counts().to_string())
print("\n── Model Comparison ───────────────────────────")
print(comparison_df.to_string())


## 7. SHAP Feature Importance

In [ ]:
shap_summary = aggregate_shap_records(shap_records, features)

print("── Top 15 Features by Average SHAP Contribution ──")
print(
    shap_summary.head(15)[
        ["Feature", "NHITSContribution", "XGBContribution",
         "BNNContribution", "AvgContribution", "Cumulative"]
    ].to_string(index=False)
)


## 8. Publication Figures

In [ ]:
saved_paths = plot_all(
    results_df    = results_df,
    risk_df       = risk_df,
    comparison_df = comparison_df,
    shap_summary  = shap_summary,
    output_dir    = "../figures",
    show          = True,          # set False for headless runs
)
for p in saved_paths:
    print("Saved:", p)


## 9. Export Results

In [ ]:
os.makedirs("../results", exist_ok=True)

results_df.to_csv("../results/model_accuracy.csv",      index=False)
risk_df.to_csv("../results/risk_assessment.csv",         index=False)
regional_summary.to_csv("../results/regional_summary.csv", index=False)
comparison_df.to_csv("../results/model_comparison.csv")

if len(shap_summary) > 0:
    shap_summary.to_csv("../results/shap_summary.csv", index=False)

logger.info("All results exported to ../results/")
print("Done. Results written to ../results/")
